<a href="https://colab.research.google.com/github/whgusdn5221/comfycolab/blob/main/Colab_%EC%8B%9C%EC%9E%91%ED%95%98%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import subprocess
import time

# 1. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

# 2. GPU 상태 확인 및 PyTorch 강제 재설치 (CUDA 오류 해결)
print("\n--- 🛠️ 환경 최적화 및 PyTorch 재설치 시작 ---")
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --force-reinstall -q

import torch
if not torch.cuda.is_available():
    print("❌ 경고: GPU가 인식되지 않습니다! [런타임 유형]을 T4 GPU로 변경했는지 확인하세요.")
else:
    print(f"✅ GPU 인식 성공: {torch.cuda.get_device_name(0)}")

# 3. ComfyUI 설치 및 필수 노드 준비
%cd /content
if not os.path.exists("ComfyUI"):
    print("--- 📥 ComfyUI 설치 중 ---")
    !git clone https://github.com/comfyanonymous/ComfyUI -q
    %cd /content/ComfyUI
    !pip install -r requirements.txt -q
    !git clone https://github.com/ltdrdata/ComfyUI-Manager custom_nodes/ComfyUI-Manager -q
    # 사용자님의 모델(IPAdapter, ControlNet 등)을 위한 필수 노드
    !git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus custom_nodes/ComfyUI_IPAdapter_plus -q
    !git clone https://github.com/Fannovel16/comfyui_controlnet_aux custom_nodes/comfyui_controlnet_aux -q
    print("✅ ComfyUI 및 커스텀 노드 설치 완료")

# 4. 드라이브 모델 폴더 심볼릭 링크 (2TB 드라이브 활용)
drive_base = "/content/drive/MyDrive/ComfyUI/models"
local_base = "/content/ComfyUI/models"

# [드라이브 폴더명 : ComfyUI 내부 폴더명]
model_map = {
    "checkpoints": "checkpoints",
    "clip": "clip",
    "clip_vision": "clip_vision",
    "control": "controlnet",  # 드라이브의 control -> controlnet으로 자동 매칭
    "ipadapter": "ipadapter",
    "schp": "schp",
    "vae": "vae",
    "insightface": "insightface"
}

print("\n--- 🔗 모델 지름길(Symbolic Link) 생성 중 ---")
for d_folder, l_folder in model_map.items():
    d_path = os.path.join(drive_base, d_folder)
    l_path = os.path.join(local_base, l_folder)

    # 드라이브에 폴더가 없으면 미리 만들어줌 (관리 편의성)
    os.makedirs(d_path, exist_ok=True)

    # 기존 빈 폴더 삭제 후 드라이브 폴더와 연결
    if os.path.exists(l_path):
        !rm -rf {l_path}
    !ln -s {d_path} {l_path}
    print(f"✅ 연결됨: {d_folder} -> {l_folder}")

# 5. Cloudflare 터널링 설치 및 실행 (수정된 버전)
print("\n--- 🌐 외부 접속 링크 생성 중 ---")

# 기존에 잘못 설치된 기록이 있을 수 있으니 확실히 설치
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb
    # 만약 위 명령어로도 안 깔리면 강제로 경로 설정
    !cp /usr/local/bin/cloudflared /usr/bin/cloudflared 2>/dev/null || true

import subprocess
import time

def start_tunnel():
    if os.path.exists("/content/tunnel.log"): os.remove("/content/tunnel.log")
    # 실행 경로를 명시적으로 전달 (./cloudflared 또는 cloudflared)
    try:
        subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
                         stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
    except FileNotFoundError:
        # 경로 문제일 경우 대비
        subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
                         stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)

start_tunnel()
print("⏳ 터널 생성 중 (8초 대기)...")
time.sleep(8)

# 링크 추출 및 출력
print("\n" + "="*60)
try:
    with open("/content/tunnel.log", "r") as f:
        import re
        content = f.read()
        url = re.findall(r"https://[a-zA-Z0-9-.]+\.trycloudflare\.com", content)
        if url:
            print(f"🎉 준비 완료! 아래 주소를 클릭하세요:\n\n{url[0]}\n")
        else:
            print("❌ 링크 생성 실패. 아래 로그를 확인하세요:")
            print(content[-500:]) # 로그 마지막 500자 출력
except:
    print("❌ 터널 정보를 읽을 수 없습니다.")
print("="*60 + "\n")

# 터널 백그라운드 실행
if os.path.exists("/content/tunnel.log"): os.remove("/content/tunnel.log")
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
                 stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
time.sleep(6)

# 링크 추출 및 출력
print("\n" + "="*60)
try:
    with open("/content/tunnel.log", "r") as f:
        import re
        url = re.findall(r"https://[a-zA-Z0-9-.]+\.trycloudflare\.com", f.read())
        if url:
            print(f"🎉 ComfyUI 실행 완료! 아래 링크로 접속하세요:\n\n{url[0]}\n")
        else:
            print("❌ 링크 생성 중... 잠시 후 아래 로그에서 링크를 직접 확인하세요.")
except:
    print("❌ 터널 정보를 읽을 수 없습니다.")
print("="*60 + "\n")

# 6. ComfyUI 최종 실행
%cd /content/ComfyUI
!python main.py --listen --preview-method auto

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- 🛠️ 환경 최적화 및 PyTorch 재설치 시작 ---
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.3 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.3 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
✅ GPU 인식 성공: Tesla T4
/content

--- 🔗 모델 지름길(Symbolic Link) 생성 중 ---
✅ 연결됨: checkpoints -> checkpoints
✅ 연결됨: clip -> 